# 06 — Financial Business Case

Business case based on Model 1 tier-level marginal effects (interaction logit).
Compares blanket rollout vs targeted rollout (High + Extreme).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

df = pd.read_csv(Path("processed/df_analysis.csv"))
df["internet_usage"] = pd.Categorical(df["internet_usage"], categories=["Low", "Medium", "High", "Extreme"], ordered=True)
for col in ["tv_product", "mobile_product", "commune"]:
    df[col] = df[col].astype("category")

model_cols = [
    "churned", "has_booster", "age", "tenure",
    "internet_usage", "tv_product", "mobile_product", "commune",
]
df_clean = df[model_cols].dropna().copy()

In [ ]:
formula_interaction = (
    "churned ~ has_booster * C(internet_usage) + age + tenure + "
    "C(tv_product) + C(mobile_product) + C(commune)"
)
fit = smf.logit(formula=formula_interaction, data=df_clean).fit(disp=False)

f1 = df_clean.copy(); f0 = df_clean.copy()
f1["has_booster"] = 1; f0["has_booster"] = 0
ite = fit.predict(f1) - fit.predict(f0)
df_clean["ite_churn"] = ite

tier_effects = df_clean.groupby("internet_usage", observed=True)["ite_churn"].mean()
tier_fin = tier_effects.to_frame(name="mean")
tier_fin["prevented_rate"] = (-tier_fin["mean"]).clip(lower=0.0)

N_CUSTOMERS_TOTAL = 1_500_000
VALUE_PER_PREVENTED_CHURN = 1800.0
BOOSTER_COST = 35.0

tier_mix = df_clean["internet_usage"].value_counts(normalize=True).sort_index()
tier_fin["mix"] = tier_mix
tier_fin["population"] = tier_fin["mix"] * N_CUSTOMERS_TOTAL
tier_fin["revenue_saved_chf"] = tier_fin["population"] * tier_fin["prevented_rate"] * VALUE_PER_PREVENTED_CHURN
tier_fin["cost_chf"] = tier_fin["population"] * BOOSTER_COST
tier_fin["net_roi_chf"] = tier_fin["revenue_saved_chf"] - tier_fin["cost_chf"]

In [ ]:
scenario_df = pd.DataFrame({
    "scenario": ["Blanket (All tiers)", "Targeted (High + Extreme)"],
    "cost_chf": [blanket_cost, target_cost],
    "saved_chf": [blanket_saved, target_saved],
    "net_chf": [blanket_net, target_net],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot1 = scenario_df.melt(id_vars="scenario", value_vars=["cost_chf", "saved_chf"], var_name="metric", value_name="value")
axes[0].set_title("Cost vs Revenue Saved", fontweight="bold")
for m, c in [("cost_chf", "#9aa4b2"), ("saved_chf", "#2a9d8f")]:
    sub = plot1[plot1["metric"] == m]
    axes[0].bar(sub["scenario"], sub["value"], alpha=0.8, label=m, color=c)
axes[0].tick_params(axis="x", rotation=10)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"CHF {x/1e6:.1f}M"))
axes[0].legend()

colors = ["#d9534f" if v < 0 else "#2ca25f" for v in scenario_df["net_chf"]]
bars = axes[1].bar(scenario_df["scenario"], scenario_df["net_chf"], color=colors)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Net Profit by Scenario", fontweight="bold")
axes[1].tick_params(axis="x", rotation=10)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"CHF {x/1e6:.1f}M"))
for b, val in zip(bars, scenario_df["net_chf"]):
    axes[1].text(b.get_x() + b.get_width()/2, val, f"{val/1e6:+.1f}M", ha="center", va="bottom" if val >= 0 else "top", fontsize=10)

fig.suptitle("Executive View: Blanket vs Targeted SuperBooster Strategy", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()